# Swarm trace generator — Colab GPU runbook

Runs **only** `swarm/generate.py` on a Colab **T4** to produce the frozen corpus.
This is the GENERATOR half: it runs **once** and writes `frozen_corpus.jsonl` +
`run_manifest.json` to your Google Drive. The local pipeline (labeling, Adaption,
eval) then reads only that frozen file and never re-invokes the swarm.

**Runtime → Change runtime type → GPU (T4)** before running.

In [ ]:
# 1. GPU check
!nvidia-smi -L

In [ ]:
# 2. Dependencies for the worker model (Colab only; not needed for the local pipeline)
!pip -q install "transformers>=4.44" accelerate bitsandbytes pyyaml

In [ ]:
# 3. Mount Drive (the frozen corpus is written here so it survives the runtime)
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# 4. Get the repo onto the Colab machine. Either clone it:
#   !git clone <your-repo-url> /content/swarm_alignment
# or upload the project folder and point PROJECT at it:
import os, sys
PROJECT = '/content/swarm_alignment'
assert os.path.isdir(PROJECT), f'upload/clone the repo to {PROJECT} first'
sys.path.insert(0, PROJECT)
os.chdir(PROJECT)

In [ ]:
# 5. Flip to the REAL worker model + target Drive. Everything else stays in config.
from common.util import load_config
cfg = load_config('configs/default.yaml')
cfg['worker']['use_stub'] = False                      # <-- real 4-bit model
cfg['worker']['model_name'] = 'Qwen/Qwen2.5-1.5B-Instruct'
cfg['worker']['quantization'] = '4bit'

drive_root = '/content/drive/MyDrive/swarm_alignment'
os.makedirs(drive_root, exist_ok=True)
cfg['paths']['frozen_corpus'] = os.path.join(drive_root, 'frozen_corpus.jsonl')
cfg['paths']['run_manifest'] = os.path.join(drive_root, 'run_manifest.json')
print('seed', cfg['seed'], '| model', cfg['worker']['model_name'], '| quant', cfg['worker']['quantization'])

In [ ]:
# 6. (Optional) tiny dry run first to sanity-check volume + token estimate.
import copy
dry = copy.deepcopy(cfg)
dry['swarm'].update(n_agents=4, rounds_per_episode=3, n_episodes=1)
dry['paths']['frozen_corpus'] = '/content/dry_corpus.jsonl'
dry['paths']['run_manifest'] = '/content/dry_manifest.json'
from swarm.generate import generate
_ = generate(dry)

In [ ]:
# 7. Full generation run -> writes the frozen corpus + manifest to Drive.
from swarm.generate import generate
summary = generate(cfg)
summary

In [ ]:
# 8. Preview + manifest (copy frozen_corpus.jsonl to your laptop's artifacts/ next).
import json
print(json.dumps(json.load(open(cfg['paths']['run_manifest'])), indent=2))
print('--- first 3 messages ---')
for i, line in enumerate(open(cfg['paths']['frozen_corpus'], encoding='utf-8')):
    if i >= 3:
        break
    print(line.strip())